# ETL: Entregable 1

In [37]:
from dotenv import dotenv_values
from sqlalchemy import create_engine, text
from sqlalchemy.engine.url import URL
from sqlalchemy import orm as sa_orm
import datetime
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyOAuth
import redshift_connector
from helpers import check_if_valid_data

In [14]:
config = dotenv_values(".env")

In [15]:
# Spotify settings
CLIENT_ID = config["CLIENT_ID"]
CLIENT_SECRET = config["CLIENT_SECRET"]
SPOTIPY_REDIRECT_URI = config["SPOTIPY_REDIRECT_URI"]
SCOPE = config["SCOPE"]

In [96]:
# DB settings
TABLENAME = config["TABLENAME"]
DB_USERNAME = config["DB_USERNAME"]
DB_PASSWORD = config["DB_PASSWORD"]
DB_HOST = config["DB_HOST"]
DB_PORT = config["DB_PORT"]
DB_NAME = config["DB_NAME"]
DB_SCHEMA = config["DB_SCHEMA"]

In [16]:
# Conexión a Spotify
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=CLIENT_ID,
                                               client_secret=CLIENT_SECRET,
                                               redirect_uri=SPOTIPY_REDIRECT_URI,
                                               scope=SCOPE))

# Conviertir una timestamp de unix en milisegundos
today = datetime.datetime.now()
yesterday = today - datetime.timedelta(days=1)
yesterday = yesterday.replace(hour=0, minute=0, second=0, microsecond=0)
yesterday_unix_timestamp = int(yesterday.timestamp()) * 1000
limit = 50    

# Retorna un diccionario con las canciones escuchadas
raw_data = sp.current_user_recently_played(limit=limit, after=yesterday_unix_timestamp)

In [17]:
raw_data.keys()

dict_keys(['items', 'next', 'cursors', 'limit', 'href'])

In [18]:
song_names = []
artist_names = []
played_at_list = []
timestamps = []

In [19]:
# Guardamos la data en listas
for song in raw_data["items"]:
    song_names.append(song["track"]["name"])
    artist_names.append(song["track"]["artists"][0]["name"])
    played_at_list.append(song["played_at"]),
    timestamps.append(song["played_at"][0:10])

In [20]:
# Creamos un diccionario de listas
song_dict = {
    "song_name" : song_names,
    "artist_name": artist_names,
    "played_at" : played_at_list,
    "timestamp" : timestamps
}

In [21]:
# Crear el DataFrame
song_df = pd.DataFrame(song_dict, columns = ["song_name",
                                             "artist_name",
                                             "played_at",
                                             "timestamp"])

In [22]:
song_df.head()

,song_name,artist_name,played_at,timestamp
0,Pink as Floyd,Red Hot Chili Peppers,2023-05-28T13:14:12.916Z,2023-05-28
1,I Want To Break Free - Single Remix,Queen,2023-05-28T13:09:19.087Z,2023-05-28
2,Hey,Pixies,2023-05-28T13:05:01.360Z,2023-05-28
3,One,U2,2023-05-28T13:01:30.596Z,2023-05-28
4,Just,Radiohead,2023-05-28T12:56:53.783Z,2023-05-28


In [23]:
# Eliminar fechas diferentes a las que queremos
clean_song_df = song_df[pd.to_datetime(song_df["played_at"]).dt.date == yesterday.date()]

In [39]:
clean_song_df

,song_name,artist_name,played_at,timestamp
20,Live Forever,Oasis,2023-05-27T23:58:19.451Z,2023-05-27
21,Love Gun,KISS,2023-05-27T23:53:42.501Z,2023-05-27
22,Song 2 - 2012 Remaster,Blur,2023-05-27T23:50:23.596Z,2023-05-27
23,Crazy Little Thing Called Love - Remastered 2011,Queen,2023-05-27T23:48:22.299Z,2023-05-27
24,Poison Heart,Ramones,2023-05-27T23:45:38.901Z,2023-05-27
25,Throwaway,Mick Jagger,2023-05-27T23:41:35.023Z,2023-05-27
26,California,Lenny Kravitz,2023-05-27T23:36:31.024Z,2023-05-27
27,Supersonic - Remastered,Oasis,2023-05-27T23:33:53.981Z,2023-05-27
28,This Picture,Placebo,2023-05-27T23:29:09.982Z,2023-05-27
29,High and Dry,Radiohead,2023-05-27T23:25:36.241Z,2023-05-27


In [38]:
# Data validation
if check_if_valid_data(clean_song_df):
    print("Data valid, proceed to Load stage...")

Data valid, proceed to Load stage...


# Conexión a PostgreSQL en local

In [ ]:
# # Load

# rows_imported = 0

# conn_string = f"postgresql+psycopg2://{DB_USERNAME}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
# engine = create_engine(conn_string)
# conn = engine.connect()

# sql_query = text(
# """
# CREATE TABLE IF NOT EXISTS coder.my_tracks_history(
#     id_my_tracks_history SERIAL NOT NULL,
#     song_name VARCHAR(200) NOT NULL,
#     artist_name VARCHAR(200) NOT NULL,
#     played_at TIMESTAMP NOT NULL,
#     timestamp DATE NOT NULL,

#     CONSTRAINT pk_my_tracks_history PRIMARY KEY (played_at)
# );
# """)

# conn.execute(sql_query)
# conn.commit()

# print(f"Importing rows {rows_imported} to {rows_imported + clean_song_df.shape[0]}... for table {TABLENAME}")

# clean_song_df.to_sql(TABLENAME,
#                con=engine,
#                schema=DB_SCHEMA,
#                index=False,
#                if_exists="append")

# rows_imported += clean_song_df.shape[0]
# print(f"Importing rows {rows_imported} to {clean_song_df.shape[0]}... for table {TABLENAME}")
# print(f"Data imported successful")

# conn.close()

# Conexión a AWS Redshift

In [40]:
# Load

# AWS Redshift settings
REDSHIFT_HOST = config["AWS_HOST"]
REDSHIFT_PORT = int(config["AWS_PORT"])
REDSHIFT_DATABASE = config["AWS_DB_NAME"]
REDSHIFT_SCHEMA = config["AWS_DB_SCHEMA"]
REDSHIFT_USERNAME = config["AWS_USER"]
REDSHIFT_PASS = config["AWS_PASSWORD"]

In [41]:
engine = create_engine(f"redshift+psycopg2://{REDSHIFT_USERNAME}:{REDSHIFT_PASS}@{REDSHIFT_HOST}:{REDSHIFT_PORT}/{REDSHIFT_DATABASE}")
conn = engine.connect()

In [42]:
sql_query = """
CREATE TABLE IF NOT EXISTS lautaropoletto_coderhouse.my_tracks_history(
    id_my_tracks_history bigint identity(0, 1) NOT NULL,
    song_name VARCHAR(200) NOT NULL,
    artist_name VARCHAR(200) NOT NULL,
    played_at TIMESTAMP NOT NULL,
    timestamp DATE NOT NULL,
    primary key(played_at))
    distkey(id_my_tracks_history)
    compound sortkey(timestamp, artist_name);
"""

conn.execute(sql_query)

C:\Users\lautaro.poletto\AppData\Local\Temp\ipykernel_1488\2042616853.py:13: RemovedIn20Warning: Deprecated API features detected! These feature(s) are not compatible with SQLAlchemy 2.0. To prevent incompatible upgrades prior to updating applications, ensure requirements files are pinned to "sqlalchemy<2.0". Set environment variable SQLALCHEMY_WARN_20=1 to show all deprecation warnings.  Set environment variable SQLALCHEMY_SILENCE_UBER_WARNING=1 to silence this message. (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  conn.execute(sql_query)


In [44]:
rows_imported = 0
table_name = "my_tracks_history"

print(f"Importing rows {rows_imported} to {rows_imported + clean_song_df.shape[0]}... for table {table_name}")

clean_song_df.to_sql(table_name,
               con=engine,
               schema=REDSHIFT_SCHEMA,
               index=False,
               if_exists="append")

rows_imported += clean_song_df.shape[0]
print(f"Importing rows {rows_imported} to {clean_song_df.shape[0]}... for table {table_name}")
print(f"Data imported successful")

conn.close()

Importing rows 0 to 22... for table my_tracks_history
Importing rows 22 to 22... for table my_tracks_history
Data imported successful
